In [13]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tarnet import Tarnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/RERUM")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
sys.path.append("..")
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [14]:
%time men_train_df = pd.read_csv(r"../dataset/Hillstrom/Men/train_men.csv", skipinitialspace=True)
%time men_test_df = pd.read_csv(r"../dataset/Hillstrom/Men/test_men.csv", skipinitialspace=True)
%time men_val_df = pd.read_csv(r"../dataset/Hillstrom/Men/val_men.csv", skipinitialspace=True)

"""Hillstrom Women"""
%time women_train_df = pd.read_csv(r"../dataset/Hillstrom/Women/train_women.csv", skipinitialspace=True)
%time women_test_df = pd.read_csv(r"../dataset/Hillstrom/Women/test_women.csv", skipinitialspace=True)
%time women_val_df = pd.read_csv(r"../dataset/Hillstrom/Women/val_women.csv", skipinitialspace=True)

# Normalize header whitespace to avoid KeyError on column lookup
for _df in [men_train_df, men_test_df, men_val_df, women_train_df, women_test_df, women_val_df]:
    _df.columns = _df.columns.str.strip()

print("Men columns:", men_train_df.columns.tolist())
print("Women columns:", women_train_df.columns.tolist())

CPU times: user 13.3 ms, sys: 4.04 ms, total: 17.4 ms
Wall time: 16.6 ms
CPU times: user 4.51 ms, sys: 1 ms, total: 5.51 ms
Wall time: 5.35 ms
CPU times: user 2.02 ms, sys: 0 ns, total: 2.02 ms
Wall time: 2.03 ms
CPU times: user 9.92 ms, sys: 2.02 ms, total: 11.9 ms
Wall time: 11.9 ms
CPU times: user 4.31 ms, sys: 1.01 ms, total: 5.33 ms
Wall time: 5.28 ms
CPU times: user 2.11 ms, sys: 0 ns, total: 2.11 ms
Wall time: 2.11 ms
Men columns: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'treatment', 'spend', 'conversion', 'visit']
Women columns: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'treatment', 'spend', 'conversion', 'visit']


In [15]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel']
label_feature = ['spend']
treatment_feature = ['treatment']

In [16]:
# 1. Define feature groups
cat_features = ['zip_code', 'channel', 'history_segment', 'mens', 'womens', 'newbie']
num_features = ['recency', 'history']

def prepare_tensors(df):
    """Convert DataFrame into separate categorical/numerical feature tensors and targets."""
    X_cat = torch.LongTensor(df[cat_features].to_numpy(copy=True))
    X_num = torch.FloatTensor(df[num_features].to_numpy(copy=True))

    t = torch.FloatTensor(df['treatment'].to_numpy(copy=True)).unsqueeze(1)
    y = torch.FloatTensor(df['spend'].to_numpy(copy=True)).unsqueeze(1)

    return X_cat, X_num, t, y

# 2. Build tensors for Men split
men_train_cat, men_train_num, men_t_train, men_y_train = prepare_tensors(men_train_df)
men_val_cat, men_val_num, men_t_val, men_y_val = prepare_tensors(men_val_df)
men_test_cat, men_test_num, men_t_test, men_y_test = prepare_tensors(men_test_df)

# 3. Build TensorDataset objects
men_train_dataset = TensorDataset(men_train_cat, men_train_num, men_t_train, men_y_train)
men_val_dataset = TensorDataset(men_val_cat, men_val_num, men_t_val, men_y_val)
men_test_dataset = TensorDataset(men_test_cat, men_test_num, men_t_test, men_y_test)

# 4. Build DataLoaders
batch_size = 1024
men_train_loader = DataLoader(men_train_dataset, batch_size=batch_size, shuffle=True)
men_val_loader = DataLoader(men_val_dataset, batch_size=batch_size, shuffle=False)
men_test_loader = DataLoader(men_test_dataset, batch_size=batch_size, shuffle=False)

# 5. Men model shape info
men_cat_dims = [men_train_df[col].nunique() for col in cat_features]
men_num_count = len(num_features)

print(f"Men model: {len(men_cat_dims)} embedding layers and {men_num_count}")

Men model: 6 embedding layers and 2


In [19]:
def show_dataset_shapes(prefix):
    print(f"\n=== {prefix.upper()} ===")
    tensor_names = [
        f"{prefix}_train_cat", f"{prefix}_train_num", f"{prefix}_t_train", f"{prefix}_y_train",
        f"{prefix}_val_cat", f"{prefix}_val_num", f"{prefix}_t_val", f"{prefix}_y_val",
        f"{prefix}_test_cat", f"{prefix}_test_num", f"{prefix}_t_test", f"{prefix}_y_test",
    ]
    
    for name in tensor_names:
        obj = globals().get(name, None)
        if obj is not None:
            print(f"{name:<20} shape={tuple(obj.shape)} dtype={obj.dtype}")
    
    dataset_names = [f"{prefix}_train_dataset", f"{prefix}_val_dataset", f"{prefix}_test_dataset"]
    for name in dataset_names:
        ds = globals().get(name, None)
        if ds is not None:
            sample_shapes = [tuple(t.shape) for t in ds.tensors]
            print(f"{name:<20} len={len(ds)} tensor_shapes={sample_shapes}")


for group in ["men", "women"]:
    if globals().get(f"{group}_train_dataset") is not None:
        show_dataset_shapes(group)


=== MEN ===
men_train_cat        shape=(25567, 6) dtype=torch.int64
men_train_num        shape=(25567, 2) dtype=torch.float32
men_t_train          shape=(25567, 1) dtype=torch.float32
men_y_train          shape=(25567, 1) dtype=torch.float32
men_val_cat          shape=(4262, 6) dtype=torch.int64
men_val_num          shape=(4262, 2) dtype=torch.float32
men_t_val            shape=(4262, 1) dtype=torch.float32
men_y_val            shape=(4262, 1) dtype=torch.float32
men_test_cat         shape=(12784, 6) dtype=torch.int64
men_test_num         shape=(12784, 2) dtype=torch.float32
men_t_test           shape=(12784, 1) dtype=torch.float32
men_y_test           shape=(12784, 1) dtype=torch.float32
men_train_dataset    len=25567 tensor_shapes=[(25567, 6), (25567, 2), (25567, 1), (25567, 1)]
men_val_dataset      len=4262 tensor_shapes=[(4262, 6), (4262, 2), (4262, 1), (4262, 1)]
men_test_dataset     len=12784 tensor_shapes=[(12784, 6), (12784, 2), (12784, 1), (12784, 1)]

=== WOMEN ===
women_tra

In [11]:
# Hillstrom Men: evaluate selected config on test set
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
men_all_runs = []

for SEED in seeds:
    seed_everything(SEED)

    men_tarnet = Tarnet(
        cate_dims=men_cat_dims,
        num_count=men_num_count,
        epochs=30,
        learning_rate=1e-3,
        weight_decay=1e-5,
        patience=30,
        shared_hidden=200,
        outcome_hidden=100,
        outcome_dropout=0,
        shared_dropout=0,
        early_stop_metric="qini",
        early_stop_start_epoch=0
    )

    men_tarnet.fit(men_train_loader, men_val_loader)

    # Test prediction
    men_test_cat_device = men_test_cat.to(device)
    men_test_num_device = men_test_num.to(device)
    men_y0_pred, men_y1_pred = men_tarnet.predict(men_test_cat_device, men_test_num_device)

    men_uplift_pred = (men_y1_pred - men_y0_pred).detach().cpu().numpy().flatten()
    men_y_true = men_y_test.detach().cpu().numpy().flatten()
    men_t_true = men_t_test.detach().cpu().numpy().flatten()

    # ATE error
    men_ate_pred = men_uplift_pred.mean()
    men_ate_true = men_y_true[men_t_true == 1].mean() - men_y_true[men_t_true == 0].mean()

    men_all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(men_y_true, men_t_true, men_uplift_pred, bins=100, plot=False),
        'AUQC': auqc(men_y_true, men_t_true, men_uplift_pred, bins=100, plot=False),
        'Lift': lift(men_y_true, men_t_true, men_uplift_pred, h=0.3),
        'KRCC': krcc(men_y_true, men_t_true, men_uplift_pred, bins=100),
        'ATE_Err': abs(men_ate_pred - men_ate_true)
    })
    print(f"Done Men Seed {SEED}")

# Aggregate final Men test metrics
men_df_results = pd.DataFrame(men_all_runs)

print("\n" + "=" * 85)
print(f"{'HILLSTROM MEN - PER-SEED DETAILS (TEST SET)':^85}")
print("=" * 85)
print(men_df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

men_mean_res = men_df_results.drop(columns='Seed').mean()
men_std_res = men_df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'HILLSTROM MEN - TEST SUMMARY (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {men_mean_res[metric]:.4f} ± {men_std_res[metric]:.4f}")
print("=" * 85)

🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Train for 30 epochs, select model with best raw Qini score
Epoch 1/30 | Base Loss: 454.7124 | Total Loss: 454.7124 | Val Loss: 413.5914 | Val Qini: 0.7198 ⭐ NEW BEST
Epoch 2/30 | Base Loss: 454.6451 | Total Loss: 454.6451 | Val Loss: 413.6831 | Val Qini: 0.6270 
Epoch 3/30 | Base Loss: 454.6878 | Total Loss: 454.6878 | Val Loss: 413.6267 | Val Qini: 0.5086 
Epoch 4/30 | Base Loss: 457.1049 | Total Loss: 457.1049 | Val Loss: 413.8498 | Val Qini: 0.7939 ⭐ NEW BEST
Epoch 5/30 | Base Loss: 457.8053 | Total Loss: 457.8053 | Val Loss: 413.4379 | Val Qini: 0.4637 
Epoch 6/30 | Base Loss: 452.7307 | Total Loss: 452.7307 | Val Loss: 413.5806 | Val Qini: 0.7489 
Epoch 7/30 | Base Loss: 453.0112 | Total Loss: 453.0112 | Val Loss: 413.7353 | Val Qini: 0.3874 
Epoch 8/30 | Base Loss: 451.7549 | Total Loss: 451.7549 | Val Loss: 413.1205 | Val Qini: 0.7953 ⭐ NEW BEST
Epoch 9/30 | Base Loss: 454.7622 | Total 

In [12]:
# Hillstrom Women: tensor prep + evaluate selected config on test set
women_train_cat, women_train_num, women_t_train, women_y_train = prepare_tensors(women_train_df)
women_val_cat, women_val_num, women_t_val, women_y_val = prepare_tensors(women_val_df)
women_test_cat, women_test_num, women_t_test, women_y_test = prepare_tensors(women_test_df)

women_train_dataset = TensorDataset(women_train_cat, women_train_num, women_t_train, women_y_train)
women_val_dataset = TensorDataset(women_val_cat, women_val_num, women_t_val, women_y_val)
women_test_dataset = TensorDataset(women_test_cat, women_test_num, women_t_test, women_y_test)

women_train_loader = DataLoader(women_train_dataset, batch_size=batch_size, shuffle=True)
women_val_loader = DataLoader(women_val_dataset, batch_size=batch_size, shuffle=False)
women_test_loader = DataLoader(women_test_dataset, batch_size=batch_size, shuffle=False)

women_cat_dims = [women_train_df[col].nunique() for col in cat_features]
women_num_count = len(num_features)

women_all_runs = []

for SEED in seeds:
    seed_everything(SEED)

    women_tarnet = Tarnet(
        cate_dims=women_cat_dims,
        num_count=women_num_count,
        epochs=30,
        learning_rate=1e-3,
        weight_decay=1e-5,
        patience=30,
        shared_hidden=200,
        outcome_hidden=100,
        outcome_dropout=0,
        shared_dropout=0,
        early_stop_metric="loss",
        early_stop_start_epoch=0
    )

    women_tarnet.fit(women_train_loader, women_val_loader)

    women_test_cat_device = women_test_cat.to(device)
    women_test_num_device = women_test_num.to(device)
    women_y0_pred, women_y1_pred = women_tarnet.predict(women_test_cat_device, women_test_num_device)

    women_uplift_pred = (women_y1_pred - women_y0_pred).detach().cpu().numpy().flatten()
    women_y_true = women_y_test.detach().cpu().numpy().flatten()
    women_t_true = women_t_test.detach().cpu().numpy().flatten()

    women_ate_pred = women_uplift_pred.mean()
    women_ate_true = women_y_true[women_t_true == 1].mean() - women_y_true[women_t_true == 0].mean()

    women_all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(women_y_true, women_t_true, women_uplift_pred, bins=100, plot=False),
        'AUQC': auqc(women_y_true, women_t_true, women_uplift_pred, bins=100, plot=False),
        'Lift': lift(women_y_true, women_t_true, women_uplift_pred, h=0.3),
        'KRCC': krcc(women_y_true, women_t_true, women_uplift_pred, bins=100),
        'ATE_Err': abs(women_ate_pred - women_ate_true)
    })
    print(f"Done Women Seed {SEED}")

women_df_results = pd.DataFrame(women_all_runs)

print("\n" + "=" * 85)
print(f"{'HILLSTROM WOMEN - PER-SEED DETAILS (TEST SET)':^85}")
print("=" * 85)
print(women_df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

women_mean_res = women_df_results.drop(columns='Seed').mean()
women_std_res = women_df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'HILLSTROM WOMEN - TEST SUMMARY (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {women_mean_res[metric]:.4f} ± {women_std_res[metric]:.4f}")
print("=" * 85)

🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: LOSS
📊 Early Stop Start Epoch: 1
📊 Strategy: Train for 30 epochs, select model with lowest validation loss
   Patience: 30 epochs
Epoch 1/30 | Base Loss: 339.0739 | Total Loss: 339.0739 | Val Loss: 201.2002 | Val Qini: 0.9964 ⭐ NEW BEST (lowest loss) | LR: 0.0010
Epoch 2/30 | Base Loss: 338.7245 | Total Loss: 338.7245 | Val Loss: 200.6606 | Val Qini: 1.1697 ⭐ NEW BEST (lowest loss) | LR: 0.0010
Epoch 3/30 | Base Loss: 335.6275 | Total Loss: 335.6275 | Val Loss: 200.8409 | Val Qini: 0.7000 (patience: 1/30) | LR: 0.0010
Epoch 4/30 | Base Loss: 339.0420 | Total Loss: 339.0420 | Val Loss: 200.9783 | Val Qini: 0.8274 (patience: 2/30) | LR: 0.0010
Epoch 5/30 | Base Loss: 338.1116 | Total Loss: 338.1116 | Val Loss: 200.9182 | Val Qini: 0.7593 (patience: 3/30) | LR: 0.0010
Epoch 6/30 | Base Loss: 337.8214 | Total Loss: 337.8214 | Val Loss: 200.9458 | Val Qini: 1.1130 (patience: 4/30) | LR: 0.0010
Epoch 7/30 | Base Loss: 337.2155 | Total Loss: 33